# Validation: our GPU simulation against Prof. Darío's states

Prof. Darío Agudelo's dataset `jdarioagudelog/nanodiskrandomkdmhexzkan` holds
**full spin vectors** — not the `s_z` projection the generative dataset carries,
but all three components at every one of the 5,245 in-disk sites. That makes it
a reference the simulation can be checked against directly, including the
topological charge, which needs the in-plane components and therefore cannot be
computed from `s_z` alone.

This notebook takes one of his annealing sweeps, runs our simulation at exactly
the same parameters and the same temperature ladder, and lays the two on top of
each other.

**Why it matters for the timing study.** If the GPU simulation reproduces his
states, the CPU arm no longer has to re-establish the physics — it only has to
establish a rate. That turns a 67-hour run into a short measurement, because the
cost is exactly linear in sweeps.

## What the comparison can and cannot settle

The agreement to look for is in `|M|`, `E`, `C_v`, `chi` and the **magnitude** of
the topological charge. Its *sign* is not a physical discrepancy: the file lists
spins by an index with no stated coordinate order, and reading that index as
`(z, y, x)` rather than `(z, x, y)` transposes the lattice, which flips the sign
of every plaquette's solid angle and therefore of `Q`. Measured here: the same
state reads `Q = +3.846` one way and `-3.846` the other. Until his writer code
says which, the sign is a convention, and only `|Q|` is evidence.

`C_v` and `chi` are also reported differently from the reference notebook, for a
reason explained where they are computed.

## 1 · Environment

In [ ]:
import os, subprocess, sys, glob, json, time
from pathlib import Path

if Path("/proc/driver/nvidia/version").exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "jax[cuda12]"])
    os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.90")

import jax, jax.numpy as jnp, numpy as np
import matplotlib.pyplot as plt
print("JAX", jax.__version__, "| backend:", jax.default_backend())
for d in jax.devices(): print("  ", d)

OUT = "/kaggle/working/validation" if Path("/kaggle/working").exists() else "validation"
os.makedirs(OUT, exist_ok=True)

## 2 · The simulation engine

Faithful to the project's reference notebook `00_benchmark_mc_observables.ipynb`:
the same checkerboard, the same uniform-sphere proposal, and in particular the
same **interfacial DMI** — in-plane bonds only, minus sign. That last point is
not cosmetic: on the same spin configuration, using all three axes with a `+`
sign changes the total energy by 50%.

Two terms the reference does not carry are added, because Prof. Darío's dataset
varies them: `KanS` (surface anisotropy, applied where a site has fewer than six
in-disk neighbours) and `Kan2`, read as the **second-order constant of the cubic
anisotropy expansion**, `(Kan2/So^6)(sx^2 sy^2 sz^2)` — the textbook K1/K2 series
that the naming follows. Being a single-site term, it leaves the checkerboard
exact, which a second-neighbour coupling would not.

In [ ]:
"""Monte Carlo engine for the nanodisk, faithful to the project's reference notebook.

Taken from `00_benchmark_mc_observables.ipynb` -- the same checkerboard, the same
uniform-sphere proposal, and in particular the same DMI: **in-plane bonds only
(x and y), with a minus sign**. That is interfacial DMI, the physics of a thin
film whose DM vector comes from an interface, and it is not interchangeable with
the all-axes form: on the same configuration the total energy differs by 50%.

Two terms the reference notebook does not carry are added here because the
tutor's dataset varies them:

- `KanS`, the surface anisotropy, applied where a site has fewer than six
  in-disk nearest neighbours (the coordination test used in the group's
  multi-GPU notebooks).
- `Kan2`, read as the SECOND-ORDER constant of the cubic anisotropy expansion,

      E_an = (Kan1 / So^4)(sx^2 sy^2 + sx^2 sz^2 + sy^2 sz^2)
           + (Kan2 / So^6)(sx^2 sy^2 sz^2)

  which is the textbook K1/K2 series and matches the naming. It is a single-site
  term, so unlike a second-neighbour coupling it leaves the checkerboard exact.
"""

from __future__ import annotations

import jax
import jax.numpy as jnp
import numpy as np
from jax import random


def build_geometry(Rd: float = 18.30, L: int = 5):
    """Disk mask, checkerboard sublattices and the surface mask."""
    L2 = int(Rd) + 1
    N = 2 * L2 + 1
    x = np.arange(-L2, L2 + 1)
    xg, yg = np.meshgrid(x, x, indexing="xy")
    disk2d = (xg ** 2 + yg ** 2) <= Rd ** 2
    disco = np.broadcast_to(disk2d, (L, N, N)).copy()

    zc, yc, xc = np.indices((L, N, N))
    checker = (xc + yc + zc) % 2
    even = (checker == 0) & disco
    odd = (checker == 1) & disco

    # Surface = fewer than six in-disk nearest neighbours. x and y wrap, z does
    # not, so the top and bottom layers are surface everywhere.
    surface = np.zeros_like(disco, dtype=bool)
    for z in range(L):
        for y in range(N):
            for xx in range(N):
                if not disco[z, y, xx]:
                    continue
                n = 0
                for dz, dy, dx in ((0, 0, 1), (0, 0, -1), (0, 1, 0),
                                   (0, -1, 0), (1, 0, 0), (-1, 0, 0)):
                    nz, ny, nx = z + dz, (y + dy) % N, (xx + dx) % N
                    if 0 <= nz < L and disco[nz, ny, nx]:
                        n += 1
                if n < 6:
                    surface[z, y, xx] = True
    return dict(L=L, N=N, Rd=Rd, disco=disco, even=even, odd=odd,
                surface=surface, disk2d=disk2d, n_active=int(disco.sum()),
                n_surface=int(surface.sum()))


def build_engine(geo, So=1.0):
    """Returns (substep, energy, magnetization, topological_charge, anneal)."""
    L, N = geo["L"], geo["N"]
    even_j = jnp.asarray(geo["even"][None, ..., None])
    odd_j = jnp.asarray(geo["odd"][None, ..., None])
    disco_j = jnp.asarray(geo["disco"][None, ..., None])
    surf_j = jnp.asarray(geo["surface"][None, ...])
    disk2d_j = jnp.asarray(geo["disk2d"])
    n_act = geo["n_active"]

    def _an(s, Kan1, Kan2, K1_eff):
        """Cubic anisotropy: the K1 invariant plus the K2 (sixth-order) one."""
        sx, sy, sz = s[..., 0], s[..., 1], s[..., 2]
        e = (K1_eff / So ** 4) * ((sx * sy) ** 2 + (sx * sz) ** 2 + (sy * sz) ** 2)
        return e + (Kan2 / So ** 6) * (sx * sy * sz) ** 2

    def substep(spins, key, beta, mask, p, K1_eff):
        S_xp = jnp.roll(spins, -1, 3); S_xm = jnp.roll(spins, 1, 3)
        S_yp = jnp.roll(spins, -1, 2); S_ym = jnp.roll(spins, 1, 2)
        S_zp = jnp.roll(spins, -1, 1).at[:, -1].set(0.0)
        S_zm = jnp.roll(spins, 1, 1).at[:, 0].set(0.0)

        kt, ka = random.split(key)
        tr = random.normal(kt, spins.shape)
        tr = tr / jnp.linalg.norm(tr, axis=-1, keepdims=True) * So
        tr = jnp.where(mask, tr, spins)
        dS = tr - spins

        sv = S_xp + S_xm + S_yp + S_ym + S_zp + S_zm
        dE = -p["Jex"] * jnp.sum(dS * sv, axis=-1)
        dE = dE + _an(tr, p["Kan1"], p["Kan2"], K1_eff) - _an(spins, p["Kan1"], p["Kan2"], K1_eff)
        # Interfacial DMI: in-plane bonds only, minus sign. See module docstring.
        dE = dE - p["KDM"] * (dS[..., 1] * (S_xp[..., 2] - S_xm[..., 2])
                              - dS[..., 2] * (S_xp[..., 1] - S_xm[..., 1])
                              + dS[..., 2] * (S_yp[..., 0] - S_ym[..., 0])
                              - dS[..., 0] * (S_yp[..., 2] - S_ym[..., 2]))
        dE = dE - p["Hex"] * (dS[..., 0] * jnp.cos(p["g"]) + dS[..., 2] * jnp.sin(p["g"]))

        r = random.uniform(ka, dE.shape)
        acc = ((dE < 0) | (r < jnp.exp(-dE * beta)))[..., None] & mask
        return jnp.where(acc, tr, spins)

    def energy(spins, p, K1_eff):
        """Total energy; each bond counted once via the +x, +y, +z shifts only."""
        S_xp = jnp.roll(spins, -1, 3)
        S_yp = jnp.roll(spins, -1, 2)
        S_zp = jnp.roll(spins, -1, 1).at[:, -1].set(0.0)
        E = -p["Jex"] * jnp.sum(spins * (S_xp + S_yp + S_zp), axis=-1)
        sx, sy, sz = spins[..., 0], spins[..., 1], spins[..., 2]
        E = E + _an(spins, p["Kan1"], p["Kan2"], K1_eff)
        E = E - p["KDM"] * (sy * S_xp[..., 2] - sz * S_xp[..., 1]
                            + sz * S_yp[..., 0] - sx * S_yp[..., 2])
        E = E - p["Hex"] * (sx * jnp.cos(p["g"]) + sz * jnp.sin(p["g"]))
        return jnp.sum(E * disco_j[..., 0], axis=(-3, -2, -1))

    def magnetization(spins):
        M = jnp.sum(spins * disco_j, axis=(1, 2, 3)) / n_act
        return jnp.linalg.norm(M, axis=-1), M[:, 2]

    def _omega(s1, s2, s3):
        num = jnp.sum(s1 * jnp.cross(s2, s3), axis=-1)
        den = (1.0 + jnp.sum(s1 * s2, -1) + jnp.sum(s2 * s3, -1) + jnp.sum(s3 * s1, -1))
        return 2.0 * jnp.arctan2(num, den)

    def topo_charge(spins):
        """Berg-Luscher charge of every layer: quantised, unlike a finite difference."""
        s00 = spins[..., :-1, :-1, :]; s10 = spins[..., :-1, 1:, :]
        s01 = spins[..., 1:, :-1, :];  s11 = spins[..., 1:, 1:, :]
        valid = (disk2d_j[:-1, :-1] & disk2d_j[:-1, 1:]
                 & disk2d_j[1:, :-1] & disk2d_j[1:, 1:])
        w = _omega(s00, s10, s11) + _omega(s00, s11, s01)
        return jnp.sum(jnp.where(valid, w, 0.0), axis=(-2, -1)) / (4.0 * jnp.pi)

    def anneal(spins, key, betas, p, n_therm, n_meas):
        K1_eff = jnp.where(surf_j, p["KanS"], p["Kan1"])

        def per_T(carry, beta):
            s, k = carry

            def therm(_, v):
                s, k = v
                k, k1, k2 = random.split(k, 3)
                s = substep(s, k1, beta, even_j, p, K1_eff)
                s = substep(s, k2, beta, odd_j, p, K1_eff)
                return s, k
            s, k = jax.lax.fori_loop(0, n_therm, therm, (s, k))

            b = s.shape[0]
            z0 = jnp.zeros(b)

            # C_v and chi are variances, and accumulating raw E and E^2
            # destroys them in float32. At T = 0.2 this run has <E> ~ -1.7e4, so
            # <E^2> ~ 2.9e8, where one float32 ulp is ~34 -- larger than the
            # variance itself. <E^2> - <E>^2 then comes out NEGATIVE and C_v
            # reads -437: not a small error but pure cancellation.
            #
            # Measuring the fluctuation about a reference taken once, before the
            # loop, keeps every accumulated quantity at the scale of the
            # fluctuation rather than of the total. A constant shift leaves the
            # variance unchanged, so this is exact. Both the centred moments and
            # the reference are returned, and `reduce_obs` recombines them.
            E_ref = energy(s, p, K1_eff)
            M_ref, _ = magnetization(s)

            def meas(_, v):
                s, k, sE, sE2, sM, sM2, sMz, sQ = v
                k, k1, k2 = random.split(k, 3)
                s = substep(s, k1, beta, even_j, p, K1_eff)
                s = substep(s, k2, beta, odd_j, p, K1_eff)
                dE = energy(s, p, K1_eff) - E_ref
                M, Mz = magnetization(s)
                dM = M - M_ref
                Q = topo_charge(s)[:, L // 2]
                return (s, k, sE + dE, sE2 + dE * dE,
                        sM + dM, sM2 + dM * dM, sMz + Mz, sQ + Q)

            s, k, sE, sE2, sM, sM2, sMz, sQ = jax.lax.fori_loop(
                0, n_meas, meas, (s, k, z0, z0, z0, z0, z0, z0))
            obs = jnp.stack([sE / n_meas, sE2 / n_meas,
                             sM / n_meas, sM2 / n_meas,
                             sMz / n_meas, sQ / n_meas, E_ref, M_ref])
            # One middle-layer snapshot per temperature, replica 0 only. The
            # whole batch would be 30 x 100 x 5 x 39 x 39 x 3 floats (~2.7 GB);
            # one replica's middle layer is 182 kB and is what the state-
            # transition figure needs.
            snap = s[0, L // 2, :, :, 2]
            return (s, k), (obs, snap)

        (s, _), (all_obs, snaps) = jax.lax.scan(per_T, (spins, key), betas)
        return all_obs, s, snaps

    return dict(substep=substep, energy=energy, magnetization=magnetization,
                topo_charge=topo_charge, anneal=anneal)


def init_spins(key, geo, batch, So=1.0):
    L, N = geo["L"], geo["N"]
    x = random.normal(key, (batch, L, N, N, 3))
    x = x / jnp.linalg.norm(x, axis=-1, keepdims=True) * So
    return x * jnp.asarray(geo["disco"][None, ..., None])


def reduce_obs(obs, temps, n_spins, kB=0.086173404):
    """Accumulators -> per-temperature scalars.

    `lax.scan` stacks each step's output along a NEW LEADING axis, so `anneal`
    returns `(n_T, 6, batch)` and the observable index is the SECOND axis. The
    obvious `obs[i]` indexes temperatures instead, silently mixing the two --
    and a test with six temperatures and six observables cannot tell the
    difference, since both shapes read `(6, 6, batch)`.
    """
    obs = np.asarray(obs)
    assert obs.ndim == 3 and obs.shape[1] == 8, (
        f"expected (n_T, 8, batch) from lax.scan, got {obs.shape}")
    dE, dE2, dM, dM2, Mz, Q, E_ref, M_ref = [obs[:, i] for i in range(8)]
    T = np.asarray(temps)[:, None]
    # Variances of the CENTRED quantities: both terms sit at the scale of the
    # fluctuation, so there is nothing to cancel.
    var_E = dE2 - dE ** 2
    var_M = dM2 - dM ** 2
    E = dE + E_ref
    M = dM + M_ref
    Cv = var_E / (kB * T ** 2 * n_spins)
    chi = var_M / (kB * T)
    return dict(T=np.asarray(temps),
                E=E.mean(1) / n_spins, E_sd=E.std(1) / n_spins,
                M=M.mean(1), M_sd=M.std(1),
                Mz=Mz.mean(1), Cv=Cv.mean(1), chi=chi.mean(1),
                Q=Q.mean(1), Q_sd=Q.std(1))


## 3 · Prof. Darío's sweep

Each block in the `.dat` file is a 15-field parameter line followed by one `index sx sy sz` line per site. A file holds many annealing sweeps; this reads the headers, groups them by parameter set, and takes one sweep.

In [ ]:
DAT = sorted(glob.glob("/kaggle/input/**/StatesHz_*.dat", recursive=True))
assert DAT, "attach the dataset jdarioagudelog/nanodiskrandomkdmhexzkan"
F = DAT[0]
print("reading", F)

COLS = ['No','L','rd','So','T','Jex','Jex2','Jex3','Jex4','Kan1','Kan2','KanS','Hex','kd','KDM']
N_SITES = 5245

offs, heads = [], []
pos = 0
with open(F, 'rb') as f:
    for line in f:
        if len(line.split()) == 15:
            offs.append(pos); heads.append([float(x) for x in line.split()])
        pos += len(line)
H = np.array(heads); offs = np.array(offs)
print(f"{len(H)} states, {len(np.unique(np.round(np.delete(H,[0,4],1),6),axis=0))} distinct parameter sets")

def read_state(i):
    with open(F, 'rb') as f:
        f.seek(int(offs[i])); f.readline()
        return np.array([[float(x) for x in f.readline().split()[1:]] for _ in range(N_SITES)])

# The sweep used here: the one whose coldest state is skyrmionic with the
# smallest Kan2 relative to Kan1, since Kan2 is the term our reading of the
# Hamiltonian is least sure about.
TARGET = dict(Kan1=0.177, Kan2=0.034, KanS=0.139, Hex=0.483, KDM=0.880)
theta = np.delete(H, [0, 4], axis=1)
u, inv = np.unique(np.round(theta, 6), axis=0, return_inverse=True)
grp = next(g for g in range(len(u))
           if all(abs(H[np.where(inv == g)[0][0], COLS.index(k)] - v) < 1e-3
                  for k, v in TARGET.items()))
ix = np.where(inv == grp)[0]
ix = ix[np.argsort(-H[ix, 4])]
TEMPS = H[ix, 4]
print(f"sweep: {len(ix)} temperatures, {TEMPS.max():.1f} -> {TEMPS.min():.1f} K")
print({k: float(H[ix[0], COLS.index(k)]) for k in
       ('Jex','Jex2','Jex3','Jex4','Kan1','Kan2','KanS','Hex','kd','KDM')})

In [ ]:
geo = build_geometry(Rd=float(H[ix[0], 2]), L=int(H[ix[0], 1]))
eng = build_engine(geo)
sites = np.argwhere(geo["disco"])
print(f"N={geo['N']}  active={geo['n_active']}  surface={geo['n_surface']}")
assert geo["n_active"] == N_SITES, "geometry does not match the file"

ref = {"T": [], "E": [], "M": [], "Mz": [], "Q": [], "sz": []}
for i in ix:
    V = np.zeros((geo["L"], geo["N"], geo["N"], 3))
    V[sites[:, 0], sites[:, 1], sites[:, 2]] = read_state(i)
    Vj = jnp.asarray(V[None])
    P0 = {k: float(H[i, COLS.index(k)]) for k in ('Jex','Kan1','Kan2','KanS','Hex','KDM')}
    P0["g"] = np.pi / 2
    K1 = jnp.where(jnp.asarray(geo["surface"][None]), P0["KanS"], P0["Kan1"])
    ref["T"].append(float(H[i, 4]))
    ref["E"].append(float(eng["energy"](Vj, P0, K1)[0]) / geo["n_active"])
    m, mz = eng["magnetization"](Vj)
    ref["M"].append(float(m[0])); ref["Mz"].append(float(mz[0]))
    ref["Q"].append(float(eng["topo_charge"](Vj)[0, geo["L"] // 2]))
    ref["sz"].append(V[geo["L"] // 2, :, :, 2].astype(np.float32))
ref = {k: np.array(v) for k, v in ref.items()}
print(f"reference states read: Q at the coldest point = {ref['Q'][-1]:+.3f}, |M| = {ref['M'][-1]:.4f}")

## 4 · Our simulation, same parameters, same ladder

`C_v` and `chi` are variances, and accumulating `E` and `E^2` directly destroys
them in float32. At `T = 0.2` this run has `<E> ~ -1.7e4`, so `<E^2> ~ 2.9e8`,
where one float32 ulp is about 34 — larger than the variance itself. The
difference `<E^2> - <E>^2` then comes out **negative** and `C_v` reads `-437`.
Measuring the fluctuation about a reference taken once before the measurement
loop keeps everything at the scale of the fluctuation; a constant shift leaves
the variance unchanged, so it is exact rather than an approximation.

In [ ]:
REPLICAS, N_THERM, N_MEAS, SEED = 100, 10_000, 5_000, 42

P = {k: float(H[ix[0], COLS.index(k)]) for k in ('Jex','Kan1','Kan2','KanS','Hex','KDM')}
P["g"] = np.pi / 2
betas = jnp.asarray(1.0 / (0.086173404 * TEMPS))

spins = init_spins(jax.random.PRNGKey(SEED), geo, REPLICAS)
run = jax.jit(lambda s, k: eng["anneal"](s, k, betas, P, N_THERM, N_MEAS))

t0 = time.perf_counter()
obs, final, snaps = run(spins, jax.random.PRNGKey(SEED + 1))
jax.block_until_ready((obs, final, snaps))
dt = time.perf_counter() - t0
sweeps = len(TEMPS) * (N_THERM + N_MEAS)
print(f"{dt:.1f} s for {len(TEMPS)} temperatures x {N_THERM}+{N_MEAS} sweeps "
      f"x {REPLICAS} replicas  ->  {1000*dt/sweeps:.3f} ms/sweep")

sim = reduce_obs(obs, TEMPS, geo["n_active"])
sim["sz"] = np.asarray(snaps)        # one per temperature, replica 0
print(f"simulation: Q at the coldest point = {sim['Q'][-1]:+.3f} +/- {sim['Q_sd'][-1]:.3f}, "
      f"|M| = {sim['M'][-1]:.4f}")

## 5 · Side by side

In [ ]:
import pandas as pd
tab = pd.DataFrame({
    "T": TEMPS,
    "E/N ref": ref["E"], "E/N sim": sim["E"],
    "|M| ref": ref["M"], "|M| sim": sim["M"],
    "|Q| ref": np.abs(ref["Q"]), "|Q| sim": np.abs(sim["Q"]),
    "Cv sim": sim["Cv"], "chi sim": sim["chi"],
}).set_index("T")
print(tab.to_string(float_format=lambda v: f"{v:9.4f}"))
tab.to_csv(f"{OUT}/comparison.csv")

c = -1
print(f"\n--- coldest point, T = {TEMPS[c]:.2f} K ---")
for name, a, b in (("|M|", ref["M"][c], sim["M"][c]),
                   ("Mz", ref["Mz"][c], sim["Mz"][c]),
                   ("|Q|", abs(ref["Q"][c]), abs(sim["Q"][c])),
                   ("E/N", ref["E"][c], sim["E"][c])):
    print(f"  {name:<5} reference {a:>9.4f}   simulation {b:>9.4f}   "
          f"{100*abs(b-a)/max(abs(a),1e-12):>6.2f}% apart")
print(f"\n  Q sign: reference {ref['Q'][c]:+.3f}, simulation {sim['Q'][c]:+.3f} — "
      "a transposed read of the file flips this, see the opening note")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
panels = [("E", "E / N  (meV)"), ("M", "$|M|$"), ("Mz", "$M_z$"),
          ("Q", "topological charge $Q$"), ("Cv", "$C_v$"), ("chi", "$\\chi$")]
for ax, (key, lab) in zip(axes.ravel(), panels):
    if key in ref:
        y = np.abs(ref[key]) if key == "Q" else ref[key]
        ax.plot(TEMPS, y, "o-", ms=5, color="0.25", label="Prof. Dario (states)")
    y = np.abs(sim[key]) if key == "Q" else sim[key]
    ax.plot(TEMPS, y, "s--", ms=4, color="#D55E00", label="our simulation", mfc="none")
    ax.set_xlabel("T (K)"); ax.set_ylabel(lab); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    ax.set_xlim(TEMPS.max() * 1.03, 0)
    if key in ("Cv", "chi"): ax.set_yscale("log")
axes[1][0].set_ylabel("$|Q|$  (sign is a read convention)")
fig.suptitle("Validation against Prof. Dario's states — same parameters, same ladder", y=1.0)
fig.tight_layout(); fig.savefig(f"{OUT}/observables.png", dpi=160, bbox_inches="tight")
plt.show()

## 6 · The state transition, side by side

Middle-layer $s_z$ along the anneal: his states on top, ours below, at the same temperatures.

In [ ]:
IM = dict(cmap="jet", vmin=-1, vmax=1, interpolation="nearest")
pick = np.linspace(0, len(TEMPS) - 1, 8).round().astype(int)
fig, axes = plt.subplots(2, len(pick), figsize=(2.3 * len(pick), 5.6))
for c_, i in enumerate(pick):
    im = axes[0][c_].imshow(ref["sz"][i], **IM)
    axes[1][c_].imshow(sim["sz"][i], **IM)
    axes[0][c_].set_title(f"{TEMPS[i]:.1f} K", fontsize=9)
    for r_ in (0, 1):
        axes[r_][c_].set_xticks([]); axes[r_][c_].set_yticks([])
axes[0][0].set_ylabel("Prof. Dario", fontsize=10)
axes[1][0].set_ylabel("our simulation", fontsize=10)
fig.colorbar(im, ax=axes, fraction=0.015, pad=0.01, label="$s_z$")
fig.suptitle("Middle-layer $s_z$ along the anneal, at the same temperatures", y=1.02)
fig.savefig(f"{OUT}/transition.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
np.savez_compressed(f"{OUT}/validation.npz",
                    T=TEMPS, ref_sz=ref["sz"], sim_sz=sim["sz"],
                    **{f"ref_{k}": v for k, v in ref.items() if k != "sz"},
                    **{f"sim_{k}": np.asarray(v) for k, v in sim.items() if k != "sz"})
json.dump({"params": {k: float(v) for k, v in P.items()},
           "temps": TEMPS.tolist(), "replicas": REPLICAS,
           "n_therm": N_THERM, "n_meas": N_MEAS,
           "ms_per_sweep": 1000 * dt / sweeps, "device": str(jax.devices()[0]),
           "ref_Q_cold": float(ref["Q"][-1]), "sim_Q_cold": float(sim["Q"][-1]),
           "ref_M_cold": float(ref["M"][-1]), "sim_M_cold": float(sim["M"][-1])},
          open(f"{OUT}/validation.json", "w"), indent=1)
print("saved:", sorted(os.listdir(OUT)))